In [1]:
cd ..

/home/gavrilev/DLResearchPPI/PPI Research/src_


In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
import  pandas as pd

import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim


import pandas as pd
import numpy as np
import h5py
import random
import time
import torch
from torch.utils.data import Dataset, DataLoader

In [33]:
from MAFusionPPI.MAFusionPPI_AS import choose_model_setting
from utils.MoleculeDataset_AS import TrainMoleculeDataset, EvalMoleculeDataset

In [5]:
training_df = pd.read_csv('datasets/cold_both_folds/train_fold1_5_0.9.csv')
training_df

,smiles,ppi_id,label,uniprot_id1,uniprot_id2
0,Cc1cc(/C=C/c2cccc(C(=O)Oc3ccccc3[N+](=O)[O-])c...,PPI_e2af6c9701a3,1,P02766,P02766
1,CC(=O)N[C@@H](CC(C)C)C(=O)N[C@H](C(=O)N[C@@H](...,PPI_6146f3037447,0,P62942,na
2,CC[C@@H]1/C=C(\C)C[C@H](C)C[C@H](OC)[C@H]2O[C@...,PPI_15838147a7f9,0,P06790,P06789
3,C1CCC(C(CC2CCCCN2)C2CCCCC2)CC1.O=C(O)/C=C\C(=O)O,PPI_3e913d9371c7,0,P10415,Q07812
4,O=C(Nc1ccc2oc(-c3ccccc3)cc(=O)c2c1)c1c[nH]nc1-...,PPI_12d05c728240,0,Q01196,Q13951
...,...,...,...,...,...
54319,CC(O)(c1ccc(-c2ccc(S(=O)(=O)c3ccc(N)nc3)nc2Nc2...,PPI_aae7d3a89141,0,Q06609,P51587
54320,CCC(C)(C)C(=O)C(=O)N1CCC[C@H]1C(=O)O[C@H](CCc1...,PPI_3619e47d2f49,0,Q86U86-5,P62805
54321,C/C=C/CC(C)C(O)C1C(=O)NC(CC)C(=O)N(C)CC(=O)N(C...,PPI_4a25885ad7c6,0,Q07817,P55957
54322,C[C@H](NC(=O)OCc1ccccc1)C(=O)NCC(=O)N[C@@H](CC...,PPI_e5b009cf8764,1,P25054,Q9NR80


In [6]:
device="cuda"

In [20]:
model = MAFusionPPI().to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-6)
criterion = nn.BCEWithLogitsLoss()

In [8]:
ds = TrainMoleculeDataset(training_df)

In [19]:
ds[100][0]['ppi_former_a'].shape

torch.Size([256, 128])

In [20]:
len(ds)

54324

In [34]:
model = choose_model_setting().to(device)

In [ ]:
for i in range(len(ds)):
    if ds[100][0]['ppi_former_a'].shape[0] != 256:
        print(ds[100][0]['ppi_former_a'].shape[0])

In [10]:
g = torch.Generator()
g.manual_seed(42)
train_loader = DataLoader(ds, batch_size=16, shuffle=True, num_workers=0, drop_last=True, generator=g)


In [35]:
start_time = time.time()
_ , _ , _ , _ = _train_one_epoch(model, train_loader, optimizer=optimizer, criterion=criterion, device=device)
end_time = time.time()

print(f'epo time -> {end_time - start_time}')

end, cp_embedding -> torch.Size([16, 256])
PPI embeddings shape after fuse -> torch.Size([16, 256])
end, cp_embedding -> torch.Size([16, 256])
PPI embeddings shape after fuse -> torch.Size([16, 256])
end, cp_embedding -> torch.Size([16, 256])
PPI embeddings shape after fuse -> torch.Size([16, 256])
end, cp_embedding -> torch.Size([16, 256])
PPI embeddings shape after fuse -> torch.Size([16, 256])
end, cp_embedding -> torch.Size([16, 256])
PPI embeddings shape after fuse -> torch.Size([16, 256])
end, cp_embedding -> torch.Size([16, 256])
PPI embeddings shape after fuse -> torch.Size([16, 256])
end, cp_embedding -> torch.Size([16, 256])
PPI embeddings shape after fuse -> torch.Size([16, 256])


KeyboardInterrupt: 

In [18]:
!nvidia-smi

Mon Mar 23 12:59:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1080 Ti     On  |   00000000:3B:00.0 Off |                  N/A |
| 29%   53C    P2             62W /  250W |   10611MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!kill -9 1713157

In [9]:
def _train_one_epoch(model,train_loader, optimizer, criterion, device):
    model.train()
    train_loss = 0.0
    all_labels, all_outputs = [], []

    for inputs, y in train_loader:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(**inputs)
        logits = outputs.view(-1)
        targets = y.view(-1).float()

        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        probs = torch.sigmoid(logits)
        all_labels.extend(targets.detach().cpu().numpy())
        all_outputs.extend(probs.detach().cpu().numpy())

    avg_loss = train_loss / max(1, len(train_loader))
    metrics = calc_metrics(ys_true=all_labels, ys_pred=all_outputs, true_threshold=0.5)
    
    return metrics, round(avg_loss, 5), all_labels, all_outputs


def train_model(self, fold, num_epochs, dataset, strct_dataset,
                strct_strategy, strct_aug_train,optimizer, criterion, 
                batch_size=32, device='cuda', num_workers=5, seed=42):
    
    train_dataset = TrainMoleculeDataset(ds_=dataset, struct_dataset=strct_dataset, strategy=strct_strategy, aug_train=strct_aug_train)
    val_dataset_es = EvalMoleculeDataset(ds_=val_subset, use_struct=use_struct, struct_dataset=strct_dataset, strategy=strct_strategy, eval_all_confs=False) 

    # drop_last=True in order to avoid bug when batch_size=1 in training phase (BatchNorn1d crashes when batch_size=1)
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True, generator=g, worker_init_fn=seed_worker)
    
    logger.info(f'Start training {fold} for {num_epochs} epochs !')
    for epoch in range(num_epochs):
        start_time = time.time()
        train_metric_dict, train_loss, _, _ = self._train_one_epoch(train_loader, optimizer, criterion, device)
        end_time = time.time()
        epoch_time = (end_time - start_time) / 60

        logger.info(f"Epoch {epoch+1} Time: {epoch_time:.2f} min, Train Loss: {train_loss:.5f}, Train AUC: {train_metric_dict['AUC']:.5f}, "
              f"Train AUPR: {train_metric_dict['AUPR']:.5f}, Precision: {train_metric_dict['Precision']:.5f}, Sensitivity: {train_metric_dict['Sensitivity']:.5f}, Specificity: {train_metric_dict['Specificity']:.5f}")

In [ ]:
model.heldout_val_model(fold='test_model', use_struct=True, num_epochs=20, dataset=training_df, strct_dataset='dataset1', strct_strategy='conditional',
                          strct_aug_train=False, strct_aug_eval=False, optimizer=optimizer, criterion=criterion, is_neg_smoo=False, save_probs=False,
                          batch_size=32, device="cuda", num_workers=5, seed=42)